# 04 — Common Statistical Model Development

The same interpretable model structure is fitted separately to the three study periods:

`demand_mw ~ C(hour) * C(day_type) + C(month)`

This captures hour-of-day effects, weekday/weekend differences, different hourly patterns by day type, monthly seasonality, and residual variation. The model is **descriptive/simulation-oriented**, not a causal model of COVID-19.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if not (project_root / "src" / "data_preparation.py").is_file():
    project_root = project_root.parent
if not (project_root / "src" / "data_preparation.py").is_file():
    raise FileNotFoundError("Run this notebook from the project root or notebooks directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import PERIOD_ORDER
figures_dir = project_root / "figures"
results_dir = project_root / "results"
figures_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

from statsmodels.stats.stattools import durbin_watson
from src.config import MODEL_FORMULA
from src.modeling import fit_period_model

df = pd.read_csv(project_root / "data/processed/luzon_hourly_clean.csv", parse_dates=["date", "datetime"])
bundles = {p: fit_period_model(df, p) for p in PERIOD_ORDER}
print("Common model formula:", MODEL_FORMULA)

In [ ]:
rows = []
for period, b in bundles.items():
    resid = b.model.resid
    rows.append({
        "period": period,
        "train_days": b.train["date"].nunique(),
        "test_days": b.test["date"].nunique(),
        "train_observations": len(b.train),
        "test_observations": len(b.test),
        "R_squared": b.model.rsquared,
        "adjusted_R_squared": b.model.rsquared_adj,
        "residual_SD_MW": float(np.std(resid, ddof=1)),
        "Durbin_Watson": float(durbin_watson(resid)),
        "AIC": b.model.aic,
        "BIC": b.model.bic,
    })
model_metrics = pd.DataFrame(rows)
model_metrics.round(3)

In [ ]:
plt.figure(figsize=(10, 5))
for period in PERIOD_ORDER:
    b = bundles[period]
    train = b.train.copy()
    train["fitted_mw"] = b.model.predict(train)
    actual = train.groupby("hour")["demand_mw"].mean()
    fitted = train.groupby("hour")["fitted_mw"].mean()
    plt.plot(actual.index + 1, actual.values, marker="o", markersize=3, label=f"{period} actual")
    plt.plot(fitted.index + 1, fitted.values, linestyle="--", label=f"{period} fitted")
plt.title("Historical vs Fitted Average 24-Hour Profiles (Calibration Data)")
plt.xlabel("NGCP Hour No. (1–24)")
plt.ylabel("Demand (MW)")
plt.xticks(range(1, 25))
plt.legend(ncol=2)
plt.tight_layout()
plt.savefig(figures_dir / "model_actual_vs_fitted_profiles.png", dpi=160)
plt.show()

### Residual dependence note

Electricity demand is a time series, so adjacent hours are not independent. A very low Durbin–Watson statistic indicates positive residual autocorrelation. For that reason, the Monte Carlo stage does **not** draw 24 independent hourly errors. It resamples **whole residual-day vectors**, preserving realistic within-day dependence. Standard OLS p-values are therefore not used as causal evidence.

In [ ]:
model_metrics.to_csv(results_dir / "model_metrics.csv", index=False)
print("MODEL SUMMARY")
for _, r in model_metrics.iterrows():
    print(f"- {r['period']}: R²={r['R_squared']:.3f}, adjusted R²={r['adjusted_R_squared']:.3f}, residual SD={r['residual_SD_MW']:.1f} MW, Durbin–Watson={r['Durbin_Watson']:.3f}")
print("Interpretation: R² shows how much historical hourly variation is captured by hour, day type, their interaction, and month. Durbin–Watson is expected to be low because load is serially correlated.")